# Activity 5: Joining, Merging and Concatenating DataFrames

In this activity, you'll lean how to merge, join and concatenate DataFrames to organize information into a single table.

## Submitting this activity:
1. Complete the required challenges and questions
2. Download the file as a Jupyter Notebook by clicking File -> Download -> Download .ipynb
3. Rename the file into the following format: YOURID_Classwork6.ipynb
4. Upload the file into the appropriate place in Canvas

# Joining DataFrames

The .join method joins DataFrames based on their index. There are 4 different kinds of joins:
* inner: the result has only rows that match <b>both</b> DataFrames
* outer: the result has rows from both DataFrames with Nan values where one DataFrame doesn't have the column or row
* left: the result keeps all values from the left DataFrame, and includes only the values from the right DataFrame that match the left one
* right: similar to the left join, but with all values from the right DataFrame, and only values from the left DataFrame that match the right one

By default, the join method performs a "left" join. Run the cells below to explore their results:

In [1]:
import pandas as pd

df1 = pd.DataFrame({"Col1":["A", "B", "C"]}, index=[0, 1, 5])
df2 = pd.DataFrame({"Col2":[4, 3, 2, 1]}, index=[3, 2, 1, 0])

df1, df2

(  Col1
 0    A
 1    B
 5    C,
    Col2
 3     4
 2     3
 1     2
 0     1)

In [2]:
df1.join(df2)

,Col1,Col2
0,A,1.0
1,B,2.0
5,C,NaN


Notice that the left DataFrame has index values 0, 1 and 5, and those are kept. The right DataFrame has index values 0, 1 and 6, which is why the row with index 5 contains a Nan.

In [3]:
df1.join(df2, how="right")

,Col1,Col2
3,NaN,4
2,NaN,3
1,B,2
0,A,1


Notice now how the converse process works. The right DataFrame has indices 3, 2, 1 and 0 (in that order, which is preserved), and the left DataFrame, having only indices 0, 1 and 5, can only match with the indices 0 and 1. The rest of the values are filled with Nans.

In [4]:
df1.join(df2, how="inner")

,Col1,Col2
0,A,1
1,B,2


With an "inner" join, only indices belonging to both DataFrames are kept.

In [5]:
df1.join(df2, how="outer")

,Col1,Col2
0,A,1.0
1,B,2.0
2,NaN,3.0
3,NaN,4.0
5,C,NaN


With an outer join, all indices in both the left and right DataFrames are kept, though the umatched values are Nans.

You can also achieve the results above using pd.merge, but the syntax is a little different:

In [6]:
pd.merge(df1, df2, left_index=True, right_index=True, how="outer")

,Col1,Col2
0,A,1.0
1,B,2.0
2,NaN,3.0
3,NaN,4.0
5,C,NaN


# Merging DataFrames

The .merge method is similar to the join one, except it's more general: it joins DataFrames based on a key column that should match both DataFrames. Here's an example below:

In [7]:
df_grade = pd.DataFrame({"ID":["A0123", "A0124", "A0125"], "grade":[100, 94, 85]})
df_attendance = pd.DataFrame({"student_ID":["A0123", "A0124", "A0126"], "absences":[0, 1, 2]})

print(df_grade)
print(df_attendance)

      ID  grade
0  A0123    100
1  A0124     94
2  A0125     85
  student_ID  absences
0      A0123         0
1      A0124         1
2      A0126         2


Both DataFrames contain a column for the "student ID", even though it's named differently. In this case, we want to merge together the DataFrames into one single table, where we have a column for student ID, their grades and their absences. This is a frequently necessary task since many databases are kept separated into different tables.

We can achieve our goal with:

In [8]:
pd.merge(df_grade, df_attendance, left_on="ID", right_on="student_ID", how="outer")

,ID,grade,student_ID,absences
0,A0123,100.0,A0123,0.0
1,A0124,94.0,A0124,1.0
2,A0125,85.0,NaN,NaN
3,NaN,NaN,A0126,2.0


The "left_on" lets us choose which column name the left DataFrame should math, the "right_on" argument lets us choose the column name on the right DataFrame that should match the one on the other DataFrame.

We can then specify the "how" parameter to suit our needs.


Now, what if we have two DataFrames, one where our key column contains repeated values?

In [9]:
df_products = pd.DataFrame({"product_ID":["AB1", "AB2", "AB3"],
                            "category":["Furniture", "Apparel", "Furniture"]})
df_prices = pd.DataFrame({"order_ID":["O1", "O1", "O1", "O2", "O2"],
                          "ID_product":["AB1", "AB2", "AB3", "AB2", "AB3"]})

print(df_products)
print(df_prices)

  product_ID   category
0        AB1  Furniture
1        AB2    Apparel
2        AB3  Furniture
  order_ID ID_product
0       O1        AB1
1       O1        AB2
2       O1        AB3
3       O2        AB2
4       O2        AB3


It won't return an error. It will instead create more rows than the original DataFrame had to acommodate all possibilities:

In [10]:
df_merged = pd.merge(df_products, df_prices, left_on="product_ID", right_on="ID_product",
         how="outer")

df_merged

,product_ID,category,order_ID,ID_product
0,AB1,Furniture,O1,AB1
1,AB2,Apparel,O1,AB2
2,AB2,Apparel,O2,AB2
3,AB3,Furniture,O1,AB3
4,AB3,Furniture,O2,AB3


Since we have an extra column we don't need, we can remove it using the .drop() method with the following parameters:

In [12]:
df_merged = df_merged.drop(columns="ID_product", axis=1)

df_merged

ValueError: Cannot specify both 'axis' and 'index'/'columns'

The axis parameter specifies whether you want to remove the row (0( or column (1). In this case, we set it to 1 because we want to remove a column.

# Concatenating DataFrames

Lastly, there's the pd.concat() method, which joins DataFrames together either vertically or horizontally in their current order, without matching any columns.

In [13]:
df_1 = pd.DataFrame({"ID":["A0123", "A0124", "A0125"], "grade":[100, 94, 85]})
df_2 = pd.DataFrame({"grade":[98, 100, 76], "ID":["A0126", "A0127", "A0128"]})

concatenated = pd.concat([df_1, df_2])
concatenated

,ID,grade
0,A0123,100
1,A0124,94
2,A0125,85
0,A0126,98
1,A0127,100
2,A0128,76


Notice something? Take a look at the indices. They're repeated! We literally copy-pasted two DataFrames, one on top of the other. To fix the index, we can use .reset_index() with drop=True. The drop=True parameter will remove the current index. When it's False, the current index is saved as a separate column for future reference. We don't want that here!

In [14]:
concatenated = concatenated.reset_index(drop=True)
concatenated

,ID,grade
0,A0123,100
1,A0124,94
2,A0125,85
3,A0126,98
4,A0127,100
5,A0128,76


That's much better.

## Part 2: Your turn!

Now, you'll be working with a real-life dataset: The Brazillian e-commerce Dataset. The only thing is: it doesn't actually come in one file, but many tables:
* orders: contains information on order placement and status
* orders_dates: contains information on order dates of purchase, delivery and estimated delivery times
* order_items: contains information on what products were included in every order
* customers: this table contains customer information and is split in two vertically
* products: this table contains information on products
* reviews: this table contains information on review scores per order

You see why joining, merging and concatenating can come in handy? Let's get to work!

Run the cell below to load the DataFrames into their appropriate variables to get started!

In [15]:
!git clone https://github.com/GzzDany/ADIAclasswork.git

from ADIAclasswork.autograde_act12 import *

orders, orders_dates = load_orders()
customers1, customers2 = load_customers()
products = load_products()
reviews = load_reviews()
order_items = load_order_items()


"git" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


Now, we can choose as a base table the "orders" table, once we've joined them together. In this case, let's take a look at both the order tables:
* what columns do they have?
* do they have missing values?

In [18]:
orders, orders_dates

(                               order_id                       customer_id  \
 0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
 1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
 2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
 3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
 4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
 ...                                 ...                               ...   
 99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
 99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
 99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
 99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
 99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   
 
       order_status order_purchase_timestamp  
 0        deliv

In this case, both tables share the same columns and indices, so we could join them with a simple "join", however, it's a better idea to use "merge" in case there might be inconsistencies. We need to merge them on the "order_id" column.

Both the tables have the same "order_id" column with that name, so let's do an outer merge:

In [27]:
merged = pd.merge(orders, orders_dates, left_on="order_id", right_on="order_id", how="outer")
merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00
...,...,...,...,...,...,...,...,...
96456,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18 00:00:00
96457,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01 00:00:00
96458,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10 00:00:00
96459,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25 00:00:00


We can fill in the missing values using the .fillna method, and specifying the value we want to fill it with. We can set it to 2018-12-31, for good measure.

In [29]:
merged.isna().sum()

merged.fillna(2018-12-31)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00
...,...,...,...,...,...,...,...,...
96456,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18 00:00:00
96457,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01 00:00:00
96458,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10 00:00:00
96459,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25 00:00:00


Now, let's look at our customer tables:

* how many rows and columns do they contain?
* what columns do they contain?

In [30]:
customers1, customers2

(                            customer_id                customer_unique_id  \
 0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 ...                                 ...                               ...   
 44996  ead1c51d9b4e7a887ee858e2649e4a13  726af76b08576225cf9585a3f6fdb2c3   
 44997  1a4d7b90459d1d8b4314bde0fa0384cf  8dac807f68a0cf9869ddeca081f5f809   
 44998  7a4bda17979cda408ede421d6f9e5337  3d58d1ec4b0a3513c4b9e7bc83e07921   
 44999  e02fa85f41de6f8e420b317a7e3444a4  0ac4a9460c1baf3051ffec5c9021a499   
 45000  d8ca929efa35ce0ce7386db3d0f3f7b2  14dae59b1edc9d335b0cafd0f9bada06   
 
        customer_zip_code_prefix          customer_city custom

The tables are a perfect candidate for a concatenation, given that what we need is to stack them vertically, just as they are!

To stack them vertically, we need to set axis=0 (the default behavior), since we're interested in stacking them row-by-row.

In [49]:
customers = pd.concat([customers1, customers2], axis=0)
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


Now, let's take our "orders" table and join to it the customers table. These should be a merge. Think about it... what column should we join them on? Do they have the same column name?

What kind of merge should we use? (Inner, outer, left or right?)

In [50]:
merged = pd.merge(merged, customers, left_on="customer_id", right_on="customer_id", how="outer")
merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,5f79b5b0931d63f1a42989eb65b9da6e,00012a2ce6f8dcda20d059ce98491703,delivered,2017-11-14 16:08:26,2017-11-14 16:35:32,2017-11-17 15:32:08,2017-11-28 15:41:30,2017-12-04 00:00:00,248ffe10d632bebe4f7267f1f44844c9,6273,osasco,SP
1,a44895d095d7e0702b6a162fa2dbeced,000161a058600d5901f007fab4c27140,delivered,2017-07-16 09:40:32,2017-07-16 09:55:12,2017-07-19 19:09:37,2017-07-25 18:57:33,2017-08-04 00:00:00,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,316a104623542e4d75189bb372bc5f8d,0001fd6190edaaf884bcaf3d49edf079,delivered,2017-02-28 11:06:43,2017-02-28 11:15:20,2017-03-01 15:24:20,2017-03-06 08:57:49,2017-03-22 00:00:00,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES
3,5825ce2e88d5346438686b0bba99e5ee,0002414f95344307404f0ace7a26f1d5,delivered,2017-08-16 13:09:20,2017-08-17 03:10:27,2017-08-19 11:34:29,2017-09-13 20:06:02,2017-09-14 00:00:00,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
4,0ab7fb08086d4af9141453c91878ed7a,000379cdec625522490c315e70c7a9fb,delivered,2018-04-02 13:42:17,2018-04-04 03:10:19,2018-04-04 18:11:09,2018-04-13 20:21:08,2018-04-18 00:00:00,0b83f73b19c2019e182fd552c048a22c,4841,sao paulo,SP
...,...,...,...,...,...,...,...,...,...,...,...,...
99437,814d6a3a7c0b32b2ad929ac6328124e9,fffecc9f79fd8c764f843e9951b11341,delivered,2018-03-29 16:59:26,2018-03-29 17:10:27,2018-03-31 14:29:38,2018-04-10 17:20:49,2018-04-27 00:00:00,e5794df8573fa179a90a7b797fc4b71f,95630,parobe,RS
99438,8c855550908247a7eff50281b92167a8,fffeda5b6d849fbd39689bb92087f431,delivered,2018-05-22 13:36:02,2018-05-22 13:54:37,2018-05-25 13:25:00,2018-06-08 18:03:31,2018-06-29 00:00:00,afbb5a642107cf6bb1ca68e863175f03,22461,rio de janeiro,RJ
99439,83b5fc912b2862c5046555ded1483ae9,ffff42319e9b2d713724ae527742af25,delivered,2018-06-13 16:57:05,2018-06-13 17:20:23,2018-06-15 18:52:00,2018-06-18 18:33:05,2018-06-25 00:00:00,680213db6ebd9e4f24d03280cbe10346,6754,taboao da serra,SP
99440,d0e7be325a1c986babc4e1cdb91edc03,ffffa3172527f765de70084a7e53aae8,delivered,2017-09-02 11:53:32,2017-09-02 12:05:40,2017-09-08 20:04:11,2017-09-14 19:47:40,2017-09-26 00:00:00,48fd7dec70f2b104a1d5e8c5c639102b,37130,alfenas,MG


Great! We may not be interested in the "customer_unique_id" or "customer_zip_code_prefix" columns, so let's drop them.

In [51]:
drop1 = merged.drop(columns="customer_unique_id")
drop2 = merged.drop(columns="customer_zip_code_prefix")

merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,5f79b5b0931d63f1a42989eb65b9da6e,00012a2ce6f8dcda20d059ce98491703,delivered,2017-11-14 16:08:26,2017-11-14 16:35:32,2017-11-17 15:32:08,2017-11-28 15:41:30,2017-12-04 00:00:00,248ffe10d632bebe4f7267f1f44844c9,6273,osasco,SP
1,a44895d095d7e0702b6a162fa2dbeced,000161a058600d5901f007fab4c27140,delivered,2017-07-16 09:40:32,2017-07-16 09:55:12,2017-07-19 19:09:37,2017-07-25 18:57:33,2017-08-04 00:00:00,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,316a104623542e4d75189bb372bc5f8d,0001fd6190edaaf884bcaf3d49edf079,delivered,2017-02-28 11:06:43,2017-02-28 11:15:20,2017-03-01 15:24:20,2017-03-06 08:57:49,2017-03-22 00:00:00,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES
3,5825ce2e88d5346438686b0bba99e5ee,0002414f95344307404f0ace7a26f1d5,delivered,2017-08-16 13:09:20,2017-08-17 03:10:27,2017-08-19 11:34:29,2017-09-13 20:06:02,2017-09-14 00:00:00,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
4,0ab7fb08086d4af9141453c91878ed7a,000379cdec625522490c315e70c7a9fb,delivered,2018-04-02 13:42:17,2018-04-04 03:10:19,2018-04-04 18:11:09,2018-04-13 20:21:08,2018-04-18 00:00:00,0b83f73b19c2019e182fd552c048a22c,4841,sao paulo,SP
...,...,...,...,...,...,...,...,...,...,...,...,...
99437,814d6a3a7c0b32b2ad929ac6328124e9,fffecc9f79fd8c764f843e9951b11341,delivered,2018-03-29 16:59:26,2018-03-29 17:10:27,2018-03-31 14:29:38,2018-04-10 17:20:49,2018-04-27 00:00:00,e5794df8573fa179a90a7b797fc4b71f,95630,parobe,RS
99438,8c855550908247a7eff50281b92167a8,fffeda5b6d849fbd39689bb92087f431,delivered,2018-05-22 13:36:02,2018-05-22 13:54:37,2018-05-25 13:25:00,2018-06-08 18:03:31,2018-06-29 00:00:00,afbb5a642107cf6bb1ca68e863175f03,22461,rio de janeiro,RJ
99439,83b5fc912b2862c5046555ded1483ae9,ffff42319e9b2d713724ae527742af25,delivered,2018-06-13 16:57:05,2018-06-13 17:20:23,2018-06-15 18:52:00,2018-06-18 18:33:05,2018-06-25 00:00:00,680213db6ebd9e4f24d03280cbe10346,6754,taboao da serra,SP
99440,d0e7be325a1c986babc4e1cdb91edc03,ffffa3172527f765de70084a7e53aae8,delivered,2017-09-02 11:53:32,2017-09-02 12:05:40,2017-09-08 20:04:11,2017-09-14 19:47:40,2017-09-26 00:00:00,48fd7dec70f2b104a1d5e8c5c639102b,37130,alfenas,MG


That's better.

Now, before we continue, it'll be a good idea to merge the order_items table. Take a look at it, and join it to the data:

In [52]:
merged = pd.merge(merged, order_items, left_on="order_id", right_on="order_id", how="outer")
merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013.0,campos dos goytacazes,RJ,1.0,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775.0,santa fe do sul,SP,1.0,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661.0,para de minas,MG,1.0,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,af861d436cfc08b2c2ddefd0ba074622,12952.0,atibaia,SP,1.0,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,64b576fb70d441e8f1b2d7d446e483c5,13226.0,varzea paulista,SP,1.0,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115626,NaN,ffaded93e5a1fce06363cfb6905676a1,NaN,NaN,NaN,NaN,NaN,NaN,587b326ba3bf8aa4d3e50fb1f38ea79f,6722.0,cotia,SP,NaN,NaN,NaN,NaN,NaN,NaN
115627,NaN,ffb81db92e7ac00ecfac978f673be8a6,NaN,NaN,NaN,NaN,NaN,NaN,14d46ad43ae7e3cd6944258b9840373b,49509.0,itabaiana,SE,NaN,NaN,NaN,NaN,NaN,NaN
115628,NaN,ffe7ffb7c7ae0d42808f387578426b3b,NaN,NaN,NaN,NaN,NaN,NaN,1942b890cee1b55dbf8176e925e79e07,90690.0,porto alegre,RS,NaN,NaN,NaN,NaN,NaN,NaN
115629,NaN,fffc22669ca576ae3f654ea64c8f36be,NaN,NaN,NaN,NaN,NaN,NaN,0f21adf44f13a61282678a89f6433c10,40255.0,salvador,BA,NaN,NaN,NaN,NaN,NaN,NaN


Note that now, there are multiple repetitions of the order_id, but that's just because every row contains a single product ordered with all the relevant information.

Now, lets take a look at the products table and see how we can integrate that information into our centralized DataFrame.

In [56]:
merged = pd.merge(merged, products, left_on="product_id", right_on="product_id", how="outer")
merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,f30149f4a8882a08895b6a242aa0d612,86c180c33f454b35e1596a99da3dddc4,delivered,2018-05-20 18:45:21,2018-05-20 18:58:59,2018-05-21 16:09:00,2018-06-06 22:11:56,2018-06-20 00:00:00,cd929c5ecff5fc60e9d808d33702e434,95890.0,...,101.65,18.59,perfumaria,53.0,596.0,6.0,300.0,20.0,16.0,16.0
1,f5eda0ded77c1293b04c953138c8331d,68f2b37558e27791155db34bcded5ac0,delivered,2017-12-12 19:20:28,2017-12-12 19:32:19,2017-12-20 20:12:42,2017-12-23 17:11:51,2018-01-05 00:00:00,cbbeff6b693e69511cf9d059f4b71036,14403.0,...,129.90,13.93,automotivo,56.0,752.0,4.0,1225.0,55.0,10.0,26.0
2,0bf736fd0fd5169d60de3699fcbcf986,6cd217b674e22cf568f6a2cf6060fd07,delivered,2017-12-21 16:21:47,2017-12-22 17:31:27,2018-01-02 22:27:47,2018-01-06 15:03:41,2018-01-16 00:00:00,f51fb63558e88eb3373773d106fa6880,2883.0,...,229.00,13.10,cama_mesa_banho,50.0,266.0,2.0,300.0,45.0,15.0,35.0
3,3aba44d8e554ab4bb8c09f6f78032ca8,82b838f513e00463174cc7cae7e76c1f,delivered,2018-08-10 13:24:35,2018-08-10 13:35:21,2018-08-13 14:43:00,2018-08-17 21:33:40,2018-08-27 00:00:00,4e32da06df703a2561f63e75b13f6260,95174.0,...,58.90,19.60,utilidades_domesticas,25.0,364.0,3.0,550.0,19.0,24.0,12.0
4,6f0dfb5b5398b271cc6bbd9ee263530e,8517e7c86998bf39a540087da6f115d9,delivered,2018-08-01 22:00:33,2018-08-01 22:15:19,2018-08-02 14:20:00,2018-08-07 17:38:52,2018-08-24 00:00:00,7f2dfd48dba158dbf61ba2ea631d93df,93530.0,...,58.90,19.60,utilidades_domesticas,25.0,364.0,3.0,550.0,19.0,24.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115626,NaN,ffaded93e5a1fce06363cfb6905676a1,NaN,NaN,NaN,NaN,NaN,NaN,587b326ba3bf8aa4d3e50fb1f38ea79f,6722.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115627,NaN,ffb81db92e7ac00ecfac978f673be8a6,NaN,NaN,NaN,NaN,NaN,NaN,14d46ad43ae7e3cd6944258b9840373b,49509.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115628,NaN,ffe7ffb7c7ae0d42808f387578426b3b,NaN,NaN,NaN,NaN,NaN,NaN,1942b890cee1b55dbf8176e925e79e07,90690.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115629,NaN,fffc22669ca576ae3f654ea64c8f36be,NaN,NaN,NaN,NaN,NaN,NaN,0f21adf44f13a61282678a89f6433c10,40255.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


What about the order review information? Let's incorporate that too!

In [59]:
merged = pd.merge(merged, reviews, left_on="order_id", right_on="order_id", how="outer")
merged

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013.0,...,650.0,28.0,9.0,14.0,97ca439bc427b48bc1cd7177abe71365,5.0,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775.0,...,30000.0,50.0,30.0,40.0,7b07bacd811c4117b742569b04ce3580,4.0,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661.0,...,3050.0,33.0,13.0,33.0,0c5b33dea94867d1ac402749e5438e8b,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,af861d436cfc08b2c2ddefd0ba074622,12952.0,...,200.0,16.0,10.0,15.0,f4028d019cb58564807486a6aaf33817,4.0,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,64b576fb70d441e8f1b2d7d446e483c5,13226.0,...,3750.0,35.0,40.0,30.0,940144190dcba6351888cafa43f3a3a5,5.0,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117049,NaN,ffaded93e5a1fce06363cfb6905676a1,NaN,NaN,NaN,NaN,NaN,NaN,587b326ba3bf8aa4d3e50fb1f38ea79f,6722.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117050,NaN,ffb81db92e7ac00ecfac978f673be8a6,NaN,NaN,NaN,NaN,NaN,NaN,14d46ad43ae7e3cd6944258b9840373b,49509.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117051,NaN,ffe7ffb7c7ae0d42808f387578426b3b,NaN,NaN,NaN,NaN,NaN,NaN,1942b890cee1b55dbf8176e925e79e07,90690.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117052,NaN,fffc22669ca576ae3f654ea64c8f36be,NaN,NaN,NaN,NaN,NaN,NaN,0f21adf44f13a61282678a89f6433c10,40255.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
import pandas as pd
df1 = pd.DataFrame({"ID":[1, 2, 3], "Value":[4, 5, 6]}, index=[0, 1, 2])
df2 = pd.DataFrame({"ID":[3, 4, 5], "Value":[7, 8, 9]}, index=[2, 3, 4])

pd.concat([df1, df2])

,ID,Value
0,1,4
1,2,5
2,3,6
2,3,7
3,4,8
4,5,9
